In [1]:
import pandas as pd
import numpy as np
from WindPy import w #可以在楼上调试，到机房再选择此函数
from datetime import datetime, timedelta
import os

In [2]:
now = datetime.now()

nowdate = '_' + now.strftime('%m%d')
print(nowdate)

with open('../年份信息.txt', 'r', encoding='utf-8') as f:
    year = f.readlines()[0]

print(year)

_1014
2025


In [3]:
if now.hour >= 15:
    today = now.strftime('%Y%m%d')  # 获取今日的日期字符串
else:
    today = (now - timedelta(days = 1)).strftime('%Y%m%d')   # 获取明日的日期字符串

In [4]:
print(today)

20251013


In [7]:
df = pd.read_excel(f"分析师定量数据.xlsx", sheet_name = '模板')#此时的数据无需清洗过
df['研究领域清洗版'].nunique()
df['七月一日'] = f'{year}0701'

In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2610 entries, 0 to 2609
Data columns (total 61 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   序号            2610 non-null   int64         
 1   分析师姓名         2610 non-null   object        
 2   唯一标识          2610 non-null   object        
 3   公司名称          2610 non-null   object        
 4   研究领域          2610 non-null   object        
 5   研究领域清洗版       2610 non-null   object        
 6   标的股票代码(原)     2610 non-null   object        
 7   标的股票代码(清洗)    2610 non-null   object        
 8   标的股票代码        0 non-null      float64       
 9   标的股票名称(原)     2610 non-null   object        
 10  标的股票名称        2610 non-null   object        
 11  标的股票名称(对比)    0 non-null      float64       
 12  推荐日期          2610 non-null   object        
 13  推荐日期清洗版       2610 non-null   datetime64[ns]
 14  推荐强度          2610 non-null   object        
 15  推荐强度清洗版       2610 non-null   object  

In [9]:
# 做一个切片，减少内存占用
df2 = df[['序号','研究领域清洗版','唯一标识','推荐日期清洗版','距离7月1日天数','中期观察日','长期观察日','七月一日']]
# 确保总量研究的领域不在定量评价体系内
df2 = df2[~df2['研究领域清洗版'].isin(['宏观','金融工程','固定收益','策略'])]

In [10]:
"""mask1 = df2['中期观察日'] > datetime(year,7,1)
mask2 = df2['长期观察日'] > datetime(year,7,1)
df2.loc[mask1,'中期观察日'] = datetime(year,7,1)
df2.loc[mask1,'中期观察'] = 1 
df2.loc[mask2,'长期观察日'] = datetime(year,7,1)
df2.loc[mask2,'长期观察'] = 1 #"""

"mask1 = df2['中期观察日'] > datetime(year,7,1)\nmask2 = df2['长期观察日'] > datetime(year,7,1)\ndf2.loc[mask1,'中期观察日'] = datetime(year,7,1)\ndf2.loc[mask1,'中期观察'] = 1 \ndf2.loc[mask2,'长期观察日'] = datetime(year,7,1)\ndf2.loc[mask2,'长期观察'] = 1 #"

In [11]:
a = df.groupby('研究领域清洗版')['唯一标识'].nunique()

In [12]:
print(a, sum(a))

研究领域清洗版
交通运输    15
传媒      17
公用事业    12
农林牧渔    14
医药生物    17
商贸零售    10
国防军工    14
基础化工    17
家用电器    11
建筑材料    14
建筑装饰    10
房地产     15
有色金属    12
机械设备    16
汽车      19
煤炭      11
环保       6
电力设备    20
电子      27
石油石化     7
社会服务    12
纺织服饰    14
综合       4
美容护理     5
计算机     22
轻工制造    13
通信      14
钢铁       3
银行      12
非银金融    17
食品饮料    23
Name: 唯一标识, dtype: int64 423


In [13]:
# industry = pd.read_excel("行业代码.xlsx",dtype={'ID':str})
# ind = [[] for _ in range(28)] 
# industryID = {'计算机、通信和其他电子设备制造业':ind[0],'机械设备制造业':ind[1],'电气机械和器材制造业':ind[2],'基础化工制造业':ind[3],
#               '医药制造业':ind[4],'汽车制造业':ind[5],'轻工制造业':ind[6],'非金属矿物制品业':ind[7],'金属制品业':ind[8],
#               '金属冶炼和压延加工业':ind[9],'铁路、船舶、航空航天和其他运输设备制造业':ind[10],'食品饮料制造加工业':ind[11],
#               '纺织服饰制造业':ind[12],'其他制造业':ind[13],'信息传输、软件和信息技术服务业':ind[14],'批发和零售业':ind[15],
#               '电力、热力、燃气及水生产和供应业':ind[16],'金融业':ind[17],'科学研究和技术服务业':ind[18],
#               '交通运输、仓储和邮政业':ind[19],'建筑业':ind[20],'房地产业':ind[21],'水利、环境和公共设施管理业':ind[22],
#               '采矿业':ind[23],'租赁和商务服务业':ind[24],'文化、体育和娱乐业':ind[25],'农、林、牧、渔业':ind[26],'社会服务':ind[27]}

# for index,row in industry.iterrows():
#     for key in industryID.keys():
#         if row['大类'] == key:
#             ID = row['ID']
#             industryID[key].append(ID)
# industryID

In [14]:
industry = pd.read_excel('行业代码.xlsx')
industryID = dict(zip(industry['大类'],industry['ID']))

In [15]:
industryID

{'农林牧渔': '801010.SI',
 '基础化工': '801030.SI',
 '钢铁': '801040.SI',
 '有色金属': '801050.SI',
 '电子': '801080.SI',
 '汽车': '801880.SI',
 '家用电器': '801110.SI',
 '食品饮料': '801120.SI',
 '纺织服饰': '801130.SI',
 '轻工制造': '801140.SI',
 '医药生物': '801150.SI',
 '公用事业': '801160.SI',
 '交通运输': '801170.SI',
 '房地产': '801180.SI',
 '商贸零售': '801200.SI',
 '社会服务': '801210.SI',
 '银行': '801780.SI',
 '非银金融': '801790.SI',
 '综合': '801230.SI',
 '建筑材料': '801710.SI',
 '建筑装饰': '801720.SI',
 '电力设备': '801730.SI',
 '机械设备': '801890.SI',
 '国防军工': '801740.SI',
 '计算机': '801750.SI',
 '传媒': '801760.SI',
 '通信': '801770.SI',
 '煤炭': '801950.SI',
 '石油石化': '801960.SI',
 '环保': '801970.SI',
 '美容护理': '801980.SI'}

In [17]:
# 去年-用证监会行业代码
# w.start()

# def industry_pct_change(ind,start_date,end_date):
#     wseedata = w.wsee(f'{",".join(industryID[ind])}', "sec_pq_pct_chg_avg",f"startDate={start_date};endDate={end_date};DynamicTime=1")
#     data = wseedata.Data[0]
#     #去空值
#     data = [x for x in data if x]
#     print(data)
#     try:
#         meanData = sum(data)/len(data)
#     except Exception as e:
#         print(f'{e}')
#         meanData = None
#     return meanData

In [37]:
w.start()

def industry_pct_change(ind,start_date,end_date):
    #wseedata = w.wsi(f'{",".join(industryID[ind])}', "pct_chg",f"startDate={start_date};endDate={end_date};DynamicTime=1")
    if pd.to_datetime(end_date) > pd.to_datetime(today):
        end_date = today
    wseedata1 = w.wsd(industryID[ind],"close", f"{start_date}")
    wseedata2 = w.wsd(industryID[ind],"close", f"{end_date}")
    data1 = wseedata1.Data[0][0]
    data2 = wseedata2.Data[0][0]
    try:
        pct = data2 / data1 - 1
    except:
        pct = None
        print("error")
    print(data1)
    print(data2)
    print(pct)
    return pct
    
#尝试
industry_pct_change("计算机","20240611","20240911")
industry_pct_change("计算机","20240611","20241211")
industry_pct_change("计算机","20240611","20240628")


3291.77
2904.29
-0.11771174778310756
3291.77
4896.26
0.4874246985664248
3291.77
3170.7
-0.03677960489341603


-0.03677960489341603

可以根据上方的申万指数去wind终端对一下，如果对不上（因为去年是用的是行业代码，今年用的是指数代码），就考虑换一个提取同行业回报率的方法，建议去wind的代码生成器上看看

In [31]:
import datetime
df_pctchange = df2[['序号','推荐日期清洗版','研究领域清洗版','距离7月1日天数','中期观察日','长期观察日','七月一日']].copy()
df_pctchange['推荐日期清洗版_str'] = df_pctchange['推荐日期清洗版'].dt.strftime('%Y%m%d')
df_pctchange['中期观察日_str'] = df_pctchange['中期观察日'].dt.strftime('%Y%m%d')
df_pctchange['长期观察日_str'] = df_pctchange['长期观察日'].dt.strftime('%Y%m%d')
df_pctchange

,序号,推荐日期清洗版,研究领域清洗版,距离7月1日天数,中期观察日,长期观察日,七月一日,推荐日期清洗版_str,中期观察日_str,长期观察日_str
0,1,2024-09-08,机械设备,297,2024-12-08,2025-03-08,20250701,20240908,20241208,20250308
1,2,2024-11-19,机械设备,225,2025-02-19,2025-05-19,20250701,20241119,20250219,20250519
2,3,2025-05-19,机械设备,44,2025-08-19,2025-11-19,20250701,20250519,20250819,20251119
3,4,2024-09-03,机械设备,302,2024-12-03,2025-03-03,20250701,20240903,20241203,20250303
4,5,2024-08-02,机械设备,334,2024-11-02,2025-02-02,20250701,20240802,20241102,20250202
5,6,2024-08-12,机械设备,324,2024-11-12,2025-02-12,20250701,20240812,20241112,20250212
6,7,2024-09-11,机械设备,294,2024-12-11,2025-03-11,20250701,20240911,20241211,20250311
7,8,2025-03-21,机械设备,103,2025-06-21,2025-09-21,20250701,20250321,20250621,20250921
8,9,2025-02-24,机械设备,128,2025-05-24,2025-08-24,20250701,20250224,20250524,20250824
9,10,2024-12-11,机械设备,203,2025-03-11,2025-06-11,20250701,20241211,20250311,20250611


In [32]:
industry_pct_change("医药生物","20230703","20240501")

8562.36
7825.68
-0.08603702717475092


-0.08603702717475092

In [38]:
import time
start = time.time()
i= 0
for index,row in df_pctchange.iterrows():
    i += 1
    num = row['序号']
    print(num)
    
    startdate = row['推荐日期清洗版_str']
    enddate_3 = row['中期观察日_str']
    enddate_6 = row['长期观察日_str']
    enddate_701 = row['七月一日']
    
    domain = row['研究领域清洗版']
    print(domain)
    
    annual_factor = 365/row['距离7月1日天数']  # 年化因子
    
    pctchange3 = industry_pct_change(domain,startdate,enddate_3)
    print(pctchange3)
    
    df_pctchange.loc[index,'同行业三个月回报率(原)'] = pctchange3
    try: 
        df_pctchange.loc[index,'同行业三个月年化回报率'] = pctchange3*4 # 不需要再除以100了
    except Exception as e:
        print(f'{e}没能提到')
    pctchange6 = industry_pct_change(domain,startdate,enddate_6)
    print(pctchange6)
    df_pctchange.loc[index,'同行业六个月回报率(原)'] = pctchange6
    try:
        df_pctchange.loc[index,'同行业六个月年化回报率'] = pctchange6*2
    except Exception as e:
        print(f'{e}没能提到')
    pctchange701 = industry_pct_change(domain,startdate,enddate_701)
    print(pctchange701)
    df_pctchange.loc[index,'同行业7.1回报率(原)'] = pctchange701
    try:
        df_pctchange.loc[index,'同行业7.1年化回报率'] = pctchange701*annual_factor
    except Exception as e:
        print(f'{e}没能提到')
    print(f'{i}/{len(df_pctchange)}')
    print('\n') 
    
end = time.time()
print(f'{round(end-start,3)}秒')

1
机械设备
1126.77
1627.53
0.44442077797598456
0.44442077797598456
1126.77
1802.83
0.5999982250148654
0.5999982250148654
1126.77
1659.53
0.47282054012797636
0.47282054012797636
1/2610


2
机械设备
1520.85
1688.29
0.11009632771147726
0.11009632771147726
1520.85
1661.15
0.09225104382417748
0.09225104382417748
1520.85
1659.53
0.091185850018082
0.091185850018082
2/2610


3
机械设备
1661.15
1939.18
0.1673720013243838
0.1673720013243838
error
1661.15
CWSDService: No data.
None
None
unsupported operand type(s) for *: 'NoneType' and 'int'没能提到
1661.15
1659.53
-0.0009752280046956274
-0.0009752280046956274
3/2610


4
机械设备
1161.63
1593.4
0.37169322417637285
0.37169322417637285
1161.63
1690.96
0.4556786584368515
0.4556786584368515
1161.63
1659.53
0.4286218503310002
0.4286218503310002
4/2610


5
机械设备
1209.81
1502.5
0.24193055107826855
0.24193055107826855
1209.81
1544.79
0.2768864532447244
0.2768864532447244
1209.81
1659.53
0.37172779196733385
0.37172779196733385
5/2610


6
机械设备
1172.4
1583.61
0.3507420675537358

2378.9
2391.2
0.005170456933876855
0.005170456933876855
42/2610


43
国防军工
1437.67
1574.72
0.09532785687953416
0.09532785687953416
1437.67
1805.3
0.25571236792866237
0.25571236792866237
1437.67
1662.32
0.15625978145193264
0.15625978145193264
43/2610


44
国防军工
1439.46
1809.85
0.257311769691412
0.257311769691412
error
1439.46
CWSDService: No data.
None
None
unsupported operand type(s) for *: 'NoneType' and 'int'没能提到
1439.46
1662.32
0.1548219471190584
0.1548219471190584
44/2610


45
国防军工
1523.69
1477.91
-0.030045481692470233
-0.030045481692470233
1523.69
1511.25
-0.008164390394371557
-0.008164390394371557
1523.69
1662.32
0.0909830739848656
0.0909830739848656
45/2610


46
国防军工
1486.39
1870.26
0.2582565813817368
0.2582565813817368
error
1486.39
CWSDService: No data.
None
None
unsupported operand type(s) for *: 'NoneType' and 'int'没能提到
1486.39
1662.32
0.11836059176932023
0.11836059176932023
46/2610


47
国防军工
1433.8
1526.43
0.06460454735667476
0.06460454735667476
1433.8
1812.09
0.2638373552796

8812.41
9032.42
0.024965928730052322
0.024965928730052322
8812.41
8364.52
-0.05082491622609475
-0.05082491622609475
84/2610


85
家用电器
7056.61
8458.1
0.19860669641655138
0.19860669641655138
7056.61
8703.83
0.23342936622542565
0.23342936622542565
7056.61
8364.52
0.18534537121932493
0.18534537121932493
85/2610


86
家用电器
8195.87
8688.38
0.06009246120302025
0.06009246120302025
error
8195.87
CWSDService: No data.
None
None
unsupported operand type(s) for *: 'NoneType' and 'int'没能提到
8195.87
8364.52
0.020577437172624613
0.020577437172624613
86/2610


87
家用电器
8662.73
8517.5
-0.016764922836103535
-0.016764922836103535
8662.73
8440.1
-0.02569975054053386
-0.02569975054053386
8662.73
8364.52
-0.03442448281315469
-0.03442448281315469
87/2610


88
家用电器
8413.5
7720.06
-0.08241992036607826
-0.08241992036607826
8413.5
8405.68
-0.0009294586081891598
-0.0009294586081891598
8413.5
8364.52
-0.005821596244131366
-0.005821596244131366
88/2610


89
煤炭
2627.78
2565.26
-0.023791946053322577
-0.02379194605332257

1905.25
2078.45
0.09090670515680355
0.09090670515680355
error
1905.25
CWSDService: No data.
None
None
unsupported operand type(s) for *: 'NoneType' and 'int'没能提到
1905.25
1926.16
0.010974937672221463
0.010974937672221463
124/2610


125
建筑装饰
1902.79
2078.45
0.09231707124800947
0.09231707124800947
error
1902.79
CWSDService: No data.
None
None
unsupported operand type(s) for *: 'NoneType' and 'int'没能提到
1902.79
1926.16
0.012281964904166998
0.012281964904166998
125/2610


126
建筑装饰
1913.3
2080.2
0.08723148486907428
0.08723148486907428
error
1913.3
CWSDService: No data.
None
None
unsupported operand type(s) for *: 'NoneType' and 'int'没能提到
1913.3
1926.16
0.00672137145246432
0.00672137145246432
126/2610


127
建筑装饰
1888.07
1926.16
0.020174040157409534
0.020174040157409534
1888.07
2146.06
0.1366421795802062
0.1366421795802062
1888.07
1926.16
0.020174040157409534
0.020174040157409534
127/2610


128
建筑装饰
1951.97
1917.85
-0.017479776840832684
-0.017479776840832684
1951.97
2093.59
0.07255234455447579


1654.64
1756.25
0.061409128269593216
0.061409128269593216
1654.64
1947.03
0.17670913310448189
0.17670913310448189
1654.64
1819.86
0.09985253589904741
0.09985253589904741
165/2610


166
农林牧渔
2572.81
2515.63
-0.022224727049412807
-0.022224727049412807
2572.81
2623.92
0.019865438955849912
0.019865438955849912
2572.81
2685.72
0.04388586798092353
0.04388586798092353
166/2610


167
农林牧渔
2705.92
2923.21
0.08030170884579002
0.08030170884579002
error
2705.92
CWSDService: No data.
None
None
unsupported operand type(s) for *: 'NoneType' and 'int'没能提到
2705.92
2685.72
-0.007465113528855372
-0.007465113528855372
167/2610


168
农林牧渔
2705.92
2923.21
0.08030170884579002
0.08030170884579002
error
2705.92
CWSDService: No data.
None
None
unsupported operand type(s) for *: 'NoneType' and 'int'没能提到
2705.92
2685.72
-0.007465113528855372
-0.007465113528855372
168/2610


169
农林牧渔
2591.62
2805.41
0.08249280372894163
0.08249280372894163
error
2591.62
CWSDService: No data.
None
None
unsupported operand type(s) fo

1610.71
1659.53
0.030309615014496583
0.030309615014496583
203/2610


204
机械设备
1557.97
1776.95
0.14055469617515093
0.14055469617515093
error
1557.97
CWSDService: No data.
None
None
unsupported operand type(s) for *: 'NoneType' and 'int'没能提到
1557.97
1659.53
0.06518739128481288
0.06518739128481288
204/2610


205
机械设备
1150.63
1585.78
0.3781841252183584
0.3781841252183584
1150.63
1690.96
0.4695949175668981
0.4695949175668981
1150.63
1659.53
0.44227944691169174
0.44227944691169174
205/2610


206
机械设备
1142.49
1498.9
0.31195896681809043
0.31195896681809043
1142.49
1740.83
0.523715743682658
0.523715743682658
1142.49
1659.53
0.4525553834169227
0.4525553834169227
206/2610


207
电子
3517.99
4366.36
0.24115190776551376
0.24115190776551376
3517.99
4471.29
0.27097859857475437
0.27097859857475437
3517.99
4614.85
0.311785991432608
0.311785991432608
207/2610


208
电子
3220.43
4863.75
0.5102796831479026
0.5102796831479026
3220.43
4740.13
0.47189350490462467
0.47189350490462467
3220.43
4614.85
0.43299186754

... (54 outputs left out)

error
4463.38
CWSDService: No data.
None
None
unsupported operand type(s) for *: 'NoneType' and 'int'没能提到
4463.38
4614.85
0.033936164969148974
0.033936164969148974
2482/2610


2523
电子
3009.82
4651.29
0.5453714840090105
0.5453714840090105
3009.82
4812.67
0.5989893083307307
0.5989893083307307
3009.82
4614.85
0.5332644477078363
0.5332644477078363
2483/2610


2524
医药生物
7302.5
7493.87
0.026206093803492037
0.026206093803492037
7302.5
8937.52
0.2238986648408079
0.2238986648408079
7302.5
7893.88
0.08098322492297161
0.08098322492297161
2484/2610


2525
医药生物
7438.83
7451.56
0.0017112906196270483
0.0017112906196270483
7438.83
9308.26
0.2513069931696248
0.2513069931696248
7438.83
7893.88
0.06117225423890593
0.06117225423890593
2485/2610


2526
医药生物
7811.64
7428.32
-0.04907036166541223
-0.04907036166541223
7811.64
7959.22
0.018892319666548918
0.018892319666548918
7811.64
7893.88
0.010527878908910271
0.010527878908910271
2486/2610


2527
医药生物
6472.63
8490.25
0.31171563954683035
0.31171563954683035
6

1740.38
1906.02
0.0951746170376584
0.0951746170376584
1740.38
1864.12
0.07109941507027195
0.07109941507027195
2524/2610


2567
非银金融
1781.56
2013.17
0.13000404140191746
0.13000404140191746
error
1781.56
CWSDService: No data.
None
None
unsupported operand type(s) for *: 'NoneType' and 'int'没能提到
1781.56
1864.12
0.04634140865309044
0.04634140865309044
2525/2610


2568
非银金融
1731.25
2154.74
0.24461516245487358
0.24461516245487358
error
1731.25
CWSDService: No data.
None
None
unsupported operand type(s) for *: 'NoneType' and 'int'没能提到
1731.25
1864.12
0.07674801444043311
0.07674801444043311
2526/2610


2569
电力设备
5596.4
7462.86
0.33351082838967905
0.33351082838967905
5596.4
6812.72
0.217339718390394
0.217339718390394
5596.4
7038.92
0.2577585590736904
0.2577585590736904
2527/2610


2570
电力设备
5481.51
7368.51
0.34424820897891273
0.34424820897891273
5481.51
7462.44
0.3613839982048741
0.3613839982048741
5481.51
7038.92
0.2841206163994958
0.2841206163994958
2528/2610


2571
电力设备
5537.82
7425.94
0.340

5030.9
5888.91
0.17054801327794245
0.17054801327794245
5030.9
4614.85
-0.08269892067025764
-0.08269892067025764
2564/2610


2608
电子
4740.13
4463.38
-0.05838447468740304
-0.05838447468740304
4740.13
5035.26
0.062262005472423665
0.062262005472423665
4740.13
4614.85
-0.026429654882882958
-0.026429654882882958
2565/2610


2609
电子
4339.66
5795.35
0.3354387210057932
0.3354387210057932
error
4339.66
CWSDService: No data.
None
None
unsupported operand type(s) for *: 'NoneType' and 'int'没能提到
4339.66
4614.85
0.06341280192457477
0.06341280192457477
2566/2610


2610
电子
4391.35
5541.52
0.2619171780887426
0.2619171780887426
error
4391.35
CWSDService: No data.
None
None
unsupported operand type(s) for *: 'NoneType' and 'int'没能提到
4391.35
4614.85
0.050895510492217744
0.050895510492217744
2567/2610


2611
电子
4339.66
5795.35
0.3354387210057932
0.3354387210057932
error
4339.66
CWSDService: No data.
None
None
unsupported operand type(s) for *: 'NoneType' and 'int'没能提到
4339.66
4614.85
0.06341280192457477
0.

1998.59
2161.72
0.08162254389344481
0.08162254389344481
error
1998.59
CWSDService: No data.
None
None
unsupported operand type(s) for *: 'NoneType' and 'int'没能提到
1998.59
2001.37
0.0013909806413521242
0.0013909806413521242
2605/2610


2650
房地产
2029.05
2012.38
-0.008215667430570894
-0.008215667430570894
2029.05
2377.64
0.17179961065523264
0.17179961065523264
2029.05
2001.37
-0.013641852098272622
-0.013641852098272622
2606/2610


2651
房地产
2036.89
2001.37
-0.017438349640874207
-0.017438349640874207
2036.89
2377.64
0.16728934797657202
0.16728934797657202
2036.89
2001.37
-0.017438349640874207
-0.017438349640874207
2607/2610


2652
房地产
2068.68
2008.35
-0.029163524566390153
-0.029163524566390153
2068.68
2360.97
0.14129299843378385
0.14129299843378385
2068.68
2001.37
-0.03253765686331378
-0.03253765686331378
2608/2610


2653
房地产
2123.06
1939.73
-0.08635177526777382
-0.08635177526777382
2123.06
2344.38
0.10424575848068351
0.10424575848068351
2123.06
2001.37
-0.0573182105074751
-0.057318210507475

In [39]:
df_pctchange.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 2610 entries, 0 to 2609
Data columns (total 16 columns):
序号              2610 non-null int64
推荐日期清洗版         2610 non-null datetime64[ns]
研究领域清洗版         2610 non-null object
距离7月1日天数        2610 non-null int64
中期观察日           2610 non-null datetime64[ns]
长期观察日           2610 non-null datetime64[ns]
七月一日            2610 non-null object
推荐日期清洗版_str     2610 non-null object
中期观察日_str       2610 non-null object
长期观察日_str       2610 non-null object
同行业三个月回报率(原)    2610 non-null float64
同行业三个月年化回报率     2610 non-null float64
同行业六个月回报率(原)    2049 non-null float64
同行业六个月年化回报率     2049 non-null float64
同行业7.1回报率(原)    2610 non-null float64
同行业7.1年化回报率     2610 non-null float64
dtypes: datetime64[ns](3), float64(6), int64(2), object(5)
memory usage: 426.6+ KB


In [40]:
#用于获取少数没有提到的数据，记得复查。
industry_pct_change("纺织服饰","20240530","20241130")

1475.08
1551.35
0.05170567020093819


0.05170567020093819

In [41]:
file_path = f'已提取数据{nowdate}.xlsx'
df_pctchange.to_excel(file_path,index=False)

import os
import platform
def open_file(file_path):
    if platform.system() == "Windows":
        os.startfile(file_path)
    elif platform.system() == "Darwin":
        os.system(f'open "{file_path}"')
    elif platform.system() == "Linux":
        os.system(f'xdg-open "{file_path}"')
    else:
        print(f"无法在 {platform.system()} 系统上打开文件。请手动查找并打开 {file_path}。")

open_file(file_path)